# ChaoSSM — Phase 24: dissecting the windowed Lyapunov proxy

Addresses **Reviewer 2 comment 5**: the windowed training proxy for the conditional Lyapunov
exponent overestimates the true value (manuscript Figure 9); how does the bias depend on the window
length W, and is a differentiable long-horizon estimator available that would remove it?

**Self-contained.** This notebook builds its own gated ChaoSSM cell from scratch and needs no output
from any other phase.

**The two estimators.** Along a fixed driven input sequence the cell has a conditional Jacobian
`J_t = ∂ step(h, x_t)/∂h` at each realized state. Let `a_W(s) = log σ_max(J_{s+W-1}···J_s)` be the
log top singular value of the W-step Jacobian product starting at `s`.
- **Windowed proxy** `λ̂_W = mean_s a_W(s) / W` — the finite-time top Lyapunov exponent over horizon W,
  the quantity a short training window estimates.
- **True conditional exponent** `λ` — the Benettin estimate (per-step QR reorthonormalization over a
  long horizon T), the W→∞ limit.

Because `a_W` is subadditive along the trajectory (`σ_max(AB) ≤ σ_max(A)σ_max(B)`), Fekete's lemma
gives `λ = inf_W a_W/W`, so **`λ̂_W ≥ λ` for every finite W, decreasing to λ as W grows**. The proxy
can only overestimate; the overestimate is the finite-window bias. This notebook measures that bias
as a function of W across a range of true λ (swept via the cell's gain), on the paper's actual gated
operator.

**The differentiable alternative.** The long-horizon QR accumulation is autograd-compatible. We show
it (i) matches the Benettin value and (ii) has a finite gradient w.r.t. the gain parameter, so it
could serve directly as a training objective — unlike the short-window proxy, whose bias would push
training toward a spuriously high λ target.

Runtime: ~10–20 min on a T4. Resumable.

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, field
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))

def setup_results_dir(project="ChaoSSM",phase="phase24_proxy_dissection"):
    base=None
    try:
        from google.colab import drive
        try: drive.mount("/content/drive",force_remount=False)
        except Exception as e1:
            print(f"[warn] drive.mount failed ({type(e1).__name__}); retrying force_remount=True")
            drive.mount("/content/drive",force_remount=True)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"/content/{project}_results/{phase}") if os.path.isdir("/content") \
             else os.path.abspath(f"./{project}_results/{phase}")
        print("="*78); print(f"[WARNING] Google Drive NOT mounted ({type(e).__name__}). Results -> {base}")
        print("[WARNING] Local disk, lost on recycle. Run the rescue cell or download."); print("="*78)
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=600,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)

torch 2.11.0+cu130 | device: cuda
gpu: Tesla T4
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase24_proxy_dissection


## Configuration

In [2]:
SMOKE_TEST=False

@dataclass
class Config:
    state_dim:int=64; d_model:int=64
    r_cap_chaotic:float=4.0; r_cap_periodic:float=3.4; gamma_init:float=1.0
    gamma_grid:tuple=(0.4,0.7,1.0,1.4,2.0,3.0)   # spans conditional lambda from <0 to >0
    seeds:int=5
    T:int=400; washout:int=100                    # driven-trajectory length for the true estimate
    windows:tuple=(1,2,4,8,16,32,64)              # proxy window sizes
    stride:int=4                                  # start-position stride for window averaging
    fc_length:int=400; fc_r:float=3.9             # task-like driving input (logistic)
CFG=Config()
if SMOKE_TEST:
    CFG.seeds=2; CFG.T=120; CFG.washout=30; CFG.gamma_grid=(0.7,1.4,3.0)
    CFG.windows=(1,4,16,32); CFG.fc_length=120
    print("*** SMOKE TEST — results meaningless ***")
print("gamma grid:",CFG.gamma_grid,"| windows:",CFG.windows)

gamma grid: (0.4, 0.7, 1.0, 1.4, 2.0, 3.0) | windows: (1, 2, 4, 8, 16, 32, 64)


## The gated ChaoSSM cell (verbatim from the trained-model harness)

In [3]:
def triangle_fold(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)
def arm_is_chaossm(a): return a.startswith("chaossm")
def arm_is_periodic(a): return "periodic" in a

class ChaoticMap(nn.Module):
    def __init__(self,N,r_cap,gamma_init=1.0):
        super().__init__(); self.r_cap=float(r_cap)
        self.w_r=nn.Parameter(torch.zeros(N))
        W=torch.empty(N,N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        self.w_gamma=nn.Parameter(torch.tensor(float(math.log(math.expm1(gamma_init)))))
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self,h):
        r=self.get_r(); u=h@self.W_mix.T+r*h*(1-h); return triangle_fold(self.get_gamma()*u)

class ChaoSSMCell(nn.Module):
    def __init__(self,cfg,arm="chaossm_chaotic"):
        super().__init__(); self.N=cfg.state_dim; self.is_chaossm=arm_is_chaossm(arm)
        r_cap=cfg.r_cap_periodic if arm_is_periodic(arm) else cfg.r_cap_chaotic
        self.phi=ChaoticMap(cfg.state_dim,r_cap,cfg.gamma_init)
        self.A=nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9,1.1)))
        self.B=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
        self.W_g=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
        self.b_g=nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self,h,x):
        g=torch.sigmoid(x@self.W_g.T+self.b_g)
        raw=g*self.phi(h@self.A.T)+(1-g)*h+x@self.B.T
        return triangle_fold(raw)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)
    def set_gamma(self,val):
        with torch.no_grad():
            self.phi.w_gamma.copy_(torch.tensor(float(math.log(math.expm1(max(val,1e-3))))))

## Driving input, realized trajectory, and the stack of conditional Jacobians

In [4]:
def make_drive(cfg,encoder,seed):
    """A task-like input sequence (logistic map) encoded into d_model, fixed per seed."""
    g=torch.Generator(device="cpu"); g.manual_seed(seed)
    x0=0.1+0.8*torch.rand(1,generator=g); xs=[x0]
    for _ in range(cfg.fc_length): xs.append(cfg.fc_r*xs[-1]*(1-xs[-1]))
    u=torch.stack(xs)[:cfg.fc_length].to(DEVICE)                 # (T,1)
    with torch.no_grad(): return encoder(u)                     # (T, d_model)

from torch.func import jacrev
@torch.no_grad()
def conditional_jacobians(cell,drive):
    """Roll the cell along the fixed drive; return realized states and per-step dJ = d step/d h."""
    T=drive.shape[0]; h=cell.init_state(1,DEVICE,torch.float32).squeeze(0); Js=[]
    for t in range(T):
        xt=drive[t]
        J=jacrev(lambda z: cell.step(z.unsqueeze(0),xt.unsqueeze(0)).squeeze(0))(h)
        Js.append(J); h=cell.step(h.unsqueeze(0),xt.unsqueeze(0)).squeeze(0)
    return torch.stack(Js)                                       # (T,N,N)

@torch.no_grad()
def true_lambda_benettin(Js,washout):
    """Top conditional Lyapunov exponent: per-step power iteration + renormalization (W->inf)."""
    T,N,_=Js.shape; q=torch.randn(N,device=Js.device); q=q/q.norm(); acc=0.0; n=0
    for t in range(T):
        v=Js[t]@q; r=v.norm(); q=v/(r+1e-12)
        if t>=washout: acc+=math.log(r.item()+1e-12); n+=1
    return acc/max(n,1)

@torch.no_grad()
def proxy_lambda_windowed(Js,W,washout,stride):
    """Finite-window proxy: mean over start positions of (1/W) log sigma_max(J_{s+W-1}...J_s)."""
    T=Js.shape[0]; vals=[]
    for s in range(washout,T-W+1,stride):
        P=Js[s]
        for i in range(1,W): P=Js[s+i]@P
        sv=torch.linalg.svdvals(P)[0]
        vals.append(math.log(sv.item()+1e-12)/W)
    return float(np.mean(vals)) if vals else float("nan")

## Differentiable long-horizon estimator (autograd through the QR accumulation)

In [5]:
def lambda_differentiable(cell,drive,washout):
    """Long-horizon top-exponent estimate built WITH autograd, so d(lambda)/d(gamma) exists.
    Returns (value_tensor, grad_wrt_w_gamma)."""
    cell.phi.w_gamma.grad=None
    T=drive.shape[0]; h=cell.init_state(1,DEVICE,torch.float32).squeeze(0)
    q=torch.randn(cell.N,device=DEVICE); q=q/q.norm(); acc=torch.zeros((),device=DEVICE); n=0
    for t in range(T):
        xt=drive[t]
        # Jacobian-vector product d step/d h applied to q, with grad to parameters retained
        hh=h.detach().requires_grad_(True)
        out=cell.step(hh.unsqueeze(0),xt.unsqueeze(0)).squeeze(0)
        Jq=torch.autograd.grad(out,hh,grad_outputs=q,create_graph=True)[0]   # adjoint J^T q; top exponent of forward and adjoint cocycle coincide
        r=Jq.norm()
        if t>=washout: acc=acc+torch.log(r+1e-12); n+=1
        q=(Jq/(r+1e-12)).detach()
        with torch.no_grad(): h=cell.step(h.unsqueeze(0),xt.unsqueeze(0)).squeeze(0)
    lam=acc/max(n,1)
    gpart=torch.autograd.grad(lam,cell.phi.w_gamma,retain_graph=False,allow_unused=True)[0]
    gval=float(gpart.item()) if gpart is not None else float("nan")
    return float(lam.item()),gval

## Run — sweep the gain, measure true λ, the windowed proxy at each W, and the differentiable λ

In [6]:
CKPT=os.path.join(RESULTS,"phase24_proxy.csv")
wcols=[f"proxy_W{w}" for w in CFG.windows]
COLS=["gamma","seed","true_lambda","diff_lambda","dlam_dgamma"]+wcols
if os.path.exists(CKPT):
    rows=pd.read_csv(CKPT).to_dict("records"); done={(round(r["gamma"],4),int(r["seed"])) for r in rows}
    print(f"[resume] {len(rows)} runs done")
else: rows=[]; done=set()
def _flush(): pd.DataFrame(rows)[COLS].to_csv(CKPT,index=False)

plan=[(gm,sd) for gm in CFG.gamma_grid for sd in range(CFG.seeds)]
t0=time.time()
for gm,sd in [p for p in plan if (round(p[0],4),p[1]) not in done]:
    torch.manual_seed(sd); np.random.seed(sd)
    cell=ChaoSSMCell(CFG).to(DEVICE); cell.set_gamma(gm)
    enc=nn.Linear(1,CFG.d_model).to(DEVICE)
    for p_ in enc.parameters(): p_.requires_grad_(False)
    drive=make_drive(CFG,enc,sd)
    Js=conditional_jacobians(cell,drive)
    true=true_lambda_benettin(Js,CFG.washout)
    prox={w:proxy_lambda_windowed(Js,w,CFG.washout,CFG.stride) for w in CFG.windows}
    dlam,dgrad=lambda_differentiable(cell,drive,CFG.washout)
    rows.append(dict(gamma=gm,seed=sd,true_lambda=true,diff_lambda=dlam,dlam_dgamma=dgrad,
                     **{f"proxy_W{w}":prox[w] for w in CFG.windows})); _flush()
    print(f"[P24] gamma={gm:.2f} s={sd} true={true:+.3f} diff={dlam:+.3f} "
          f"dλ/dγ={dgrad:+.3f} | proxyW1={prox[CFG.windows[0]]:+.3f} proxyW{CFG.windows[-1]}={prox[CFG.windows[-1]]:+.3f}")
RES=pd.DataFrame(rows)[COLS]; _flush()
print(f"\nwall clock: {(time.time()-t0)/60:.1f} min")

[P24] gamma=0.40 s=0 true=-0.100 diff=-0.100 dλ/dγ=+0.141 | proxyW1=-0.043 proxyW64=-0.100
[P24] gamma=0.40 s=1 true=-0.028 diff=-0.028 dλ/dγ=+0.233 | proxyW1=-0.022 proxyW64=-0.028
[P24] gamma=0.40 s=2 true=-0.120 diff=-0.118 dλ/dγ=+0.138 | proxyW1=-0.065 proxyW64=-0.118
[P24] gamma=0.40 s=3 true=-0.111 diff=-0.109 dλ/dγ=+0.075 | proxyW1=-0.047 proxyW64=-0.106
[P24] gamma=0.40 s=4 true=-0.083 diff=-0.083 dλ/dγ=+0.063 | proxyW1=-0.048 proxyW64=-0.084
[P24] gamma=0.70 s=0 true=+0.090 diff=+0.089 dλ/dγ=+0.400 | proxyW1=+0.253 proxyW64=+0.090
[P24] gamma=0.70 s=1 true=+0.139 diff=+0.139 dλ/dγ=+0.274 | proxyW1=+0.303 proxyW64=+0.141
[P24] gamma=0.70 s=2 true=+0.181 diff=+0.181 dλ/dγ=+0.614 | proxyW1=+0.267 proxyW64=+0.179
[P24] gamma=0.70 s=3 true=+0.082 diff=+0.081 dλ/dγ=+0.391 | proxyW1=+0.306 proxyW64=+0.094
[P24] gamma=0.70 s=4 true=+0.018 diff=+0.020 dλ/dγ=+0.409 | proxyW1=+0.271 proxyW64=+0.021
[P24] gamma=1.00 s=0 true=+0.295 diff=+0.295 dλ/dγ=+0.432 | proxyW1=+0.523 proxyW64=+0.298

## Analysis — the proxy's bias as a function of window length

In [7]:
g=RES.groupby("gamma")
true_mu=g.true_lambda.mean()
print("Windowed-proxy bias (proxy_W − true), averaged over seeds, per gain:\n")
hdr="gamma   true_lam  "+"  ".join(f"W={w:<4d}" for w in CFG.windows); print(hdr)
bias_by_W={w:[] for w in CFG.windows}
for gm in CFG.gamma_grid:
    sub=RES[RES.gamma==gm]; tl=sub.true_lambda.mean()
    line=f"{gm:5.2f}  {tl:+7.3f}  "
    for w in CFG.windows:
        b=sub[f"proxy_W{w}"].mean()-tl; bias_by_W[w].append(b); line+=f"{b:+6.3f} "
    print(line)
print("\nMean bias across all gains, by window length (should be >0 and decreasing):")
mb=pd.DataFrame({"W":list(CFG.windows),
                 "mean_bias":[float(np.mean(bias_by_W[w])) for w in CFG.windows],
                 "mean_proxy":[RES[f"proxy_W{w}"].mean() for w in CFG.windows]})
mb["mean_true"]=RES.true_lambda.mean(); savetable(mb,"phase24_bias_by_window.csv")
print(mb.to_string(index=False))
nonneg=all(b>-1e-3 for w in CFG.windows for b in bias_by_W[w])
decr=all(np.mean(bias_by_W[CFG.windows[i]])>=np.mean(bias_by_W[CFG.windows[i+1]])-1e-3
         for i in range(len(CFG.windows)-1))
print(f"\nproxy >= true at every W and gain: {nonneg}")
print(f"bias decreases monotonically with W: {decr}")
print("\nDifferentiable estimator vs Benettin true (should match; gradient should be finite):")
dd=RES.groupby("gamma").agg(true=("true_lambda","mean"),diff=("diff_lambda","mean"),
                            grad=("dlam_dgamma","mean"))
dd["|diff-true|"]=(dd["diff"]-dd["true"]).abs(); print(dd.round(4).to_string())
print(f"\nmax |differentiable − Benettin| = {dd['|diff-true|'].max():.4f} "
      f"(small => the long-horizon estimator is unbiased)")
print(f"gradient dλ/dγ finite at all gains: {bool(RES.dlam_dgamma.notna().all() and np.isfinite(RES.dlam_dgamma).all())}")
savetable(RES,"phase24_proxy.csv")

Windowed-proxy bias (proxy_W − true), averaged over seeds, per gain:

gamma   true_lam  W=1     W=2     W=4     W=8     W=16    W=32    W=64  
 0.40   -0.089  +0.043 +0.016 +0.008 +0.005 +0.004 +0.002 +0.002 
 0.70   +0.102  +0.178 +0.081 +0.036 +0.020 +0.012 +0.007 +0.003 
 1.00   +0.362  +0.198 +0.071 +0.022 +0.010 +0.005 +0.003 -0.000 
 1.40   +0.557  +0.287 +0.142 +0.061 +0.026 +0.010 +0.005 +0.002 
 2.00   +0.611  +0.494 +0.326 +0.209 +0.131 +0.066 +0.033 +0.015 
 3.00   +1.096  +0.424 +0.277 +0.155 +0.074 +0.032 +0.015 +0.013 

Mean bias across all gains, by window length (should be >0 and decreasing):
saved /content/drive/My Drive/ChaoSSM/results/phase24_proxy_dissection/phase24_bias_by_window.csv
 W  mean_bias  mean_proxy  mean_true
 1   0.270652    0.710444   0.439791
 2   0.152117    0.591909   0.439791
 4   0.082054    0.521846   0.439791
 8   0.044470    0.484261   0.439791
16   0.021426    0.461217   0.439791
32   0.010795    0.450586   0.439791
64   0.005631    0.445423  

## Figure (600 dpi)

In [8]:
fig,ax=plt.subplots(1,2,figsize=(12.5,4.7))
# (a) proxy_W vs true lambda, one line per W, with y=x
tl=[RES[RES.gamma==gm].true_lambda.mean() for gm in CFG.gamma_grid]
cmap=plt.cm.viridis(np.linspace(0,0.9,len(CFG.windows)))
for wi,w in enumerate(CFG.windows):
    pv=[RES[RES.gamma==gm][f"proxy_W{w}"].mean() for gm in CFG.gamma_grid]
    ax[0].plot(tl,pv,marker="o",ms=5,color=cmap[wi],label=f"W={w}")
lim=[min(tl)-0.05,max(tl)+0.25]
ax[0].plot(lim,lim,"k--",lw=1,label="proxy = true")
ax[0].set_xlabel(r"true conditional $\lambda$ (Benettin)"); ax[0].set_ylabel(r"windowed proxy $\hat\lambda_W$")
ax[0].set_title("(a) The proxy sits above the diagonal; longer W descends to it",fontsize=10.5)
ax[0].legend(fontsize=8,ncol=2); ax[0].grid(alpha=.25)
# (b) mean bias vs W
mbias=[float(np.mean([RES[RES.gamma==gm][f"proxy_W{w}"].mean()-RES[RES.gamma==gm].true_lambda.mean()
        for gm in CFG.gamma_grid])) for w in CFG.windows]
ax[1].plot(CFG.windows,mbias,marker="s",ms=7,lw=2,color="#B71C1C")
ax[1].axhline(0,ls=":",color="#333",lw=1)
ax[1].set_xscale("log",base=2); ax[1].set_xlabel("window length W (log scale)")
ax[1].set_ylabel(r"mean bias  $\hat\lambda_W-\lambda$")
ax[1].set_title("(b) Finite-window bias decays toward zero as W grows",fontsize=10.5)
ax[1].grid(alpha=.25)
fig.tight_layout(); savefig(fig,"phase24_proxy_dissection.png")

saved /content/drive/My Drive/ChaoSSM/results/phase24_proxy_dissection/phase24_proxy_dissection.png


## Rescue if Drive was not mounted

In [9]:
import shutil
if not RESULTS.startswith("/content/drive"):
    z=shutil.make_archive("/content/phase24_results","zip",RESULTS); print("archive:",z)
    try:
        from google.colab import files; files.download(z)
    except Exception as e: print("download",z,"manually:",e)
else: print("on Drive:",RESULTS)

on Drive: /content/drive/My Drive/ChaoSSM/results/phase24_proxy_dissection
